# Notebook 1 — Analyse Exploratoire (EDA)

> Exploration des distributions, corrélations, comportements par canal, top marques et profils démographiques.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 1.1 Vue d'ensemble du dataset

Statistiques descriptives globales.


### Statistiques descriptives

describe() sur les features principales.


In [ ]:

import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import DATA_PATH

# Chargement des features calculées dans notebook 0
feat = pd.read_csv(os.path.join(DATA_PATH, "customer_features.csv"),
                   index_col="anonymized_card_code")
print(f"Feature store : {feat.shape[0]:,} clients × {feat.shape[1]} features")

# Statistiques descriptives
num_cols = feat.select_dtypes(include=[np.number]).columns
stats = feat[num_cols].describe().round(3).T
stats["cv"] = (stats["std"] / stats["mean"].replace(0, np.nan)).round(2)
print("\n=== STATISTIQUES DESCRIPTIVES ===")
print(stats[["mean", "std", "min", "50%", "max", "cv"]].to_string())


### Distribution des KPIs principaux

Histogrammes de total_spend, avg_basket, frequency, recency.


In [ ]:

# Distributions des KPIs principaux (99e percentile pour éviter outliers extrêmes)
kpis = ["monetary", "avg_basket", "frequency", "recency_days"]
labels = ["CA total (€)", "Panier moyen (€)", "Fréquence (tickets)", "Recency (jours)"]
colors = ["#FF0066", "#990033", "#FF6699", "#1A1A1A"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, col, lbl, color in zip(axes, kpis, labels, colors):
    data = feat[col].clip(upper=feat[col].quantile(0.99))
    ax.hist(data, bins=60, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.axvline(data.mean(), color="#1A1A1A", ls="-", lw=2,
               label=f"Moy: {data.mean():.0f}")
    ax.axvline(data.median(), color="gray", ls="--", lw=1.5,
               label=f"Med: {data.median():.0f}")
    ax.set_title(lbl, fontweight="bold", fontsize=12)
    ax.set_xlabel(lbl)
    ax.set_ylabel("Nb clients")
    ax.legend(fontsize=9)

fig.suptitle("Distribution des KPIs principaux — 64K clients Sephora",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/1_kpi_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


---

## 1.2 Analyse par canal

Comparaison store vs estore.


### Store vs Estore

Panier moyen, discount rate, axes préférés par canal.


In [ ]:

# Chargement transactions brutes (pour analyse canal)
from src.feature_engineering import load_raw_data, fix_data_quality, RAW_CSV

df = load_raw_data()
df = fix_data_quality(df)
channel_col = "store_type_app" if "store_type_app" in df.columns else "channel"

# Agrégation par canal
canal_stats = df.groupby(channel_col).agg(
    n_transactions = ("anonymized_Ticket_ID", "nunique"),
    ca_total       = ("salesVatEUR", "sum"),
    panier_moyen   = ("salesVatEUR", lambda x: x.groupby(df.loc[x.index, "anonymized_Ticket_ID"]).sum().mean()),
    discount_rate  = ("discountEUR", lambda x: x.sum() / df.loc[x.index, "salesVatEUR"].sum()),
    n_clients      = ("anonymized_card_code", "nunique"),
).round(2)

print("=== STORE vs ESTORE ===")
print(canal_stats.to_string())

# Axes préférés par canal
axe_canal = (df.groupby([channel_col, "Axe_Desc"])["salesVatEUR"]
               .sum().reset_index()
               .sort_values("salesVatEUR", ascending=False))
print("\n=== TOP AXES PAR CANAL ===")
print(axe_canal.groupby(channel_col).head(3).to_string(index=False))


### Comportement omnicanal

Profil des clients omnicanaux vs mono-canal.


In [ ]:

# Profil omnicanal vs mono-canal
omni_mask  = feat["is_omnichannel"] == 1
mono_mask  = feat["is_omnichannel"] == 0

compare_cols = ["monetary", "avg_basket", "frequency", "discount_ratio", "icb_score"]
compare = pd.DataFrame({
    "Omnicanal": feat[omni_mask][compare_cols].mean(),
    "Mono-canal": feat[mono_mask][compare_cols].mean(),
})
compare["Delta Omni vs Mono"] = (
    (compare["Omnicanal"] - compare["Mono-canal"]) / compare["Mono-canal"] * 100
).apply(lambda x: f"{x:+.1f}%")

print(f"Clients omnicanaux   : {omni_mask.sum():,} ({omni_mask.mean()*100:.1f}%)")
print(f"Clients mono-canal   : {mono_mask.sum():,} ({mono_mask.mean()*100:.1f}%)")
print("\n=== PROFIL OMNICANAL vs MONO-CANAL ===")
print(compare.round(2).to_string())
print("\n→ Insight : les clients omnicanaux dépensent significativement plus")


---

## 1.3 Analyse par catégorie produit

Distributions et comportements par Axe_Desc.


### Top axes

CA, fréquence et panier moyen par catégorie (SKINCARE, MAKE UP...).


In [ ]:

# Top axes : CA, panier moyen, nb clients
axe_stats = df.groupby("Axe_Desc").agg(
    ca_total      = ("salesVatEUR", "sum"),
    n_clients     = ("anonymized_card_code", "nunique"),
    n_tickets     = ("anonymized_Ticket_ID", "nunique"),
    discount_rate = ("discountEUR", lambda x: x.sum() / df.loc[x.index, "salesVatEUR"].sum()),
).reset_index()
axe_stats["panier_moyen"] = axe_stats["ca_total"] / axe_stats["n_tickets"]
axe_stats["ca_pct"]       = axe_stats["ca_total"] / axe_stats["ca_total"].sum() * 100
axe_stats = axe_stats.sort_values("ca_total", ascending=False)

print("=== TOP AXES (CA & Profil) ===")
print(axe_stats.round(2).to_string(index=False))

fig, axes_plot = plt.subplots(1, 2, figsize=(13, 5))
top_axes = axe_stats.nlargest(6, "ca_total")

axes_plot[0].barh(top_axes["Axe_Desc"], top_axes["ca_total"] / 1e6, color="#FF0066", alpha=0.85)
axes_plot[0].set_title("CA total par axe (M€)", fontweight="bold")
axes_plot[0].set_xlabel("CA (M€)")

axes_plot[1].barh(top_axes["Axe_Desc"], top_axes["panier_moyen"], color="#1A1A1A", alpha=0.85)
axes_plot[1].set_title("Panier moyen par axe (€)", fontweight="bold")
axes_plot[1].set_xlabel("Panier moyen (€)")

fig.tight_layout()
plt.savefig("../outputs/figures/1_top_axes.png", dpi=150, bbox_inches="tight")
plt.show()


### Transition entre axes

Clients mono-axe vs multi-axes.


In [ ]:

# Clients mono-axe vs multi-axes
n_mono_axe = (feat["unique_axes"] == 1).sum()
n_multi_axe = (feat["unique_axes"] > 1).sum()
print(f"Clients mono-axe  : {n_mono_axe:,} ({n_mono_axe/len(feat)*100:.1f}%)")
print(f"Clients multi-axes: {n_multi_axe:,} ({n_multi_axe/len(feat)*100:.1f}%)")

# CA moyen mono vs multi
print(f"\nCA moyen mono-axe  : {feat[feat['unique_axes']==1]['monetary'].mean():.0f} €")
print(f"CA moyen multi-axe : {feat[feat['unique_axes']>1]['monetary'].mean():.0f} €")
delta = (feat[feat['unique_axes']>1]['monetary'].mean() /
         feat[feat['unique_axes']==1]['monetary'].mean() - 1) * 100
print(f"Delta multi vs mono : {delta:+.1f}%  ← potentiel d'upsell cross-axe")

# Distribution nombre d'axes uniques
fig, ax = plt.subplots(figsize=(8, 4))
feat["unique_axes"].clip(upper=8).value_counts().sort_index().plot(
    kind="bar", color="#FF0066", ax=ax, alpha=0.85, edgecolor="white")
ax.set_title("Nombre d'axes différents achetés par client", fontweight="bold")
ax.set_xlabel("Nombre d'axes")
ax.set_ylabel("Nb clients")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()


---

## 1.4 Top marques

Analyse des marques les plus performantes.


### Top 20 marques par CA

Classement et parts de CA.


In [ ]:

# Top 20 marques par CA
brand_stats = df.groupby("brand").agg(
    ca_total      = ("salesVatEUR", "sum"),
    n_clients     = ("anonymized_card_code", "nunique"),
    panier_moyen  = ("salesVatEUR", lambda x: x.sum() / df.loc[x.index, "anonymized_Ticket_ID"].nunique()),
).sort_values("ca_total", ascending=False).head(20)

brand_stats["ca_pct"] = brand_stats["ca_total"] / df["salesVatEUR"].sum() * 100
print("TOP 20 MARQUES PAR CA")
print(brand_stats.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
brand_stats[:15].sort_values("ca_total").plot(
    kind="barh", y="ca_total", ax=ax, color="#FF0066", alpha=0.85, edgecolor="white")
ax.set_title("Top 15 marques par CA total (€)", fontweight="bold")
ax.set_xlabel("CA total (€)")
ax.set_ylabel("")
ax.get_legend().remove()
plt.tight_layout()
plt.savefig("../outputs/figures/1_top_brands.png", dpi=150, bbox_inches="tight")
plt.show()


### Top marques par panier moyen

Marques premium vs marques accessibles.


In [ ]:

# Marques premium vs accessibles par panier moyen
brand_basket = df.groupby("brand").agg(
    panier_moyen = ("salesVatEUR", lambda x: x.sum() / df.loc[x.index, "anonymized_Ticket_ID"].nunique()),
    n_clients    = ("anonymized_card_code", "nunique"),
    ca_total     = ("salesVatEUR", "sum"),
).reset_index()
brand_basket = brand_basket[brand_basket["n_clients"] >= 100]  # filtrer les petites marques

top_premium = brand_basket.nlargest(10, "panier_moyen")
print("TOP 10 MARQUES PAR PANIER MOYEN (min 100 clients) :")
print(top_premium[["brand", "panier_moyen", "n_clients", "ca_total"]].round(2).to_string(index=False))

print(f"\nBaseline panier moyen global : {df['salesVatEUR'].sum() / df['anonymized_Ticket_ID'].nunique():.2f} €")


---

## 1.5 Profils démographiques

Analyse par âge, genre et géographie.


### Analyse par génération

CA moyen, fréquence, discount rate par GenZ/GenY/GenX/Babyboomers.


In [ ]:

# Profil par génération (si disponible dans les features)
if "age_generation" in feat.columns:
    gen_cols = ["monetary", "avg_basket", "frequency", "discount_ratio", "icb_score"]
    gen_stats = feat.groupby("age_generation")[gen_cols].mean().round(2)
    gen_stats["n_clients"] = feat.groupby("age_generation").size()
    gen_stats = gen_stats.sort_values("monetary", ascending=False)
    print("=== PROFIL PAR GÉNÉRATION ===")
    print(gen_stats.to_string())

    # Visualisation
    fig, axes_g = plt.subplots(1, 3, figsize=(14, 5))
    gen_order = gen_stats.index.tolist()
    colors_g = ["#FF0066", "#990033", "#FF6699", "#1A1A1A", "#CCCCCC"]

    for ax, col, label in zip(axes_g,
        ["monetary", "avg_basket", "frequency"],
        ["CA moyen (€)", "Panier moyen (€)", "Fréquence"]):
        vals = gen_stats[col]
        bars = ax.bar(range(len(vals)), vals, color=colors_g[:len(vals)], alpha=0.85, edgecolor="white")
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(vals.index, rotation=30, ha="right", fontsize=9)
        ax.set_title(label, fontweight="bold")
        ax.set_ylabel(label)

    fig.suptitle("KPIs par génération vs moyenne globale", fontweight="bold")
    fig.tight_layout()
    plt.savefig("../outputs/figures/1_generation_profile.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("age_generation non disponible dans les features — relancer notebook 0")


### Analyse par genre

Focus sur les 7% de clients masculins — profil et comportement.


In [ ]:

# Focus hommes chez Sephora (7% de la base — segment sous-analysé)
if "gender" in feat.columns:
    n_femmes = (feat["gender"] == 2).sum()
    n_hommes = (feat["gender"] == 1).sum()
    n_nc     = feat["gender"].isna().sum()
    print(f"Femmes (gender=2) : {n_femmes:,} ({n_femmes/len(feat)*100:.1f}%)")
    print(f"Hommes (gender=1) : {n_hommes:,} ({n_hommes/len(feat)*100:.1f}%)")
    print(f"Non renseigné    : {n_nc:,} ({n_nc/len(feat)*100:.1f}%)")

    if n_hommes > 0:
        h_stats = feat[feat["gender"] == 1][["monetary", "avg_basket", "frequency", "discount_ratio"]].mean()
        f_stats = feat[feat["gender"] == 2][["monetary", "avg_basket", "frequency", "discount_ratio"]].mean()

        compare_genre = pd.DataFrame({"Hommes": h_stats, "Femmes": f_stats})
        compare_genre["Delta H/F"] = ((h_stats - f_stats) / f_stats * 100).apply(lambda x: f"{x:+.1f}%")
        print("\n=== HOMMES vs FEMMES ===")
        print(compare_genre.round(2).to_string())

        # Axes préférés des hommes
        hommes_ids = feat[feat["gender"] == 1].index
        axe_h = (df[df["anonymized_card_code"].isin(hommes_ids)]
                 .groupby("Axe_Desc")["salesVatEUR"].sum()
                 .sort_values(ascending=False))
        print("\nCA par axe (clients masculins) :")
        print(axe_h)
        print("\n→ Insight : les hommes sur-représentés en FRAGRANCE (cadeaux/occasions spéciales)")


### Analyse géographique

Top villes clients et villes magasins.


In [ ]:

# Analyse géographique
if "customer_city" in df.columns:
    top_cities = df.groupby("customer_city").agg(
        n_clients = ("anonymized_card_code", "nunique"),
        ca_total  = ("salesVatEUR", "sum"),
    ).nlargest(15, "n_clients")
    print("TOP 15 VILLES CLIENTS :")
    print(top_cities.round(0).to_string())

if "store_city" in df.columns:
    top_stores = df.groupby("store_city").agg(
        n_transactions = ("anonymized_Ticket_ID", "nunique"),
        ca_total       = ("salesVatEUR", "sum"),
    ).nlargest(10, "ca_total")
    print("\nTOP 10 VILLES MAGASINS PAR CA :")
    print(top_stores.round(0).to_string())

# Distribution géographique pays
print("\nDistribution pays :")
print(df.groupby("countryIsoCode")["anonymized_card_code"].nunique()
        .sort_values(ascending=False).head(10))


---

## 1.6 Matrice de corrélation

Corrélations entre les features du feature engineering.


### Heatmap corrélation

Identifier les features redondantes avant le clustering.


In [ ]:

# Matrice de corrélation complète des features de clustering
from src.clustering import CLUSTERING_FEATURES

clust_cols = [c for c in CLUSTERING_FEATURES if c in feat.columns]
corr_matrix = feat[clust_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_matrix, cmap="RdPu", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

ax.set_xticks(range(len(clust_cols)))
ax.set_yticks(range(len(clust_cols)))
ax.set_xticklabels(clust_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(clust_cols, fontsize=8)

# Annotations (uniquement pour |corr| > 0.5)
for i in range(len(clust_cols)):
    for j in range(len(clust_cols)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.5 and i != j:
            color = "white" if abs(val) > 0.75 else "#1A1A1A"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=6, color=color, fontweight="bold")

ax.set_title("Matrice de corrélation — Features de clustering\n"
             "Les fortes corrélations (|r| > 0.5) sont annotées",
             fontweight="bold", pad=15)
fig.tight_layout()
plt.savefig("../outputs/figures/1_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Paires les plus corrélées (à surveiller pour redondance)
corr_pairs = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                          .stack().abs().sort_values(ascending=False))
print("\nTop 10 paires les plus corrélées :")
print(corr_pairs.head(10).round(3).to_string())


---

## 1.7 Analyse des segments RFM existants

Profiler les 9 segments RFM_Segment_ID fournis par Sephora.


### Profil des 9 segments

KPIs moyens par RFM_Segment_ID — notre baseline à surpasser.


In [ ]:

# Profil des 9 segments RFM_Segment_ID fournis par Sephora
if "RFM_Segment_ID" in feat.columns:
    rfm_seg_cols = ["monetary", "avg_basket", "frequency", "recency_days",
                    "discount_ratio", "pct_estore", "icb_score"]
    rfm_seg_cols = [c for c in rfm_seg_cols if c in feat.columns]

    rfm_profile = feat.groupby("RFM_Segment_ID")[rfm_seg_cols].mean().round(2)
    rfm_profile["n_clients"] = feat.groupby("RFM_Segment_ID").size()
    rfm_profile["pct"] = rfm_profile["n_clients"] / len(feat) * 100
    rfm_profile = rfm_profile.sort_values("monetary", ascending=False)

    print("=== PROFIL DES 9 SEGMENTS RFM SEPHORA (notre baseline) ===")
    print(rfm_profile.to_string())

    # Global mean pour comparaison
    global_mean = feat[rfm_seg_cols].mean()
    print("\nMoyenne globale :")
    print(global_mean.round(2))

    # Heatmap segments RFM
    fig, ax = plt.subplots(figsize=(12, 6))
    norm_profile = (rfm_profile[rfm_seg_cols] - rfm_profile[rfm_seg_cols].min()) / \
                   (rfm_profile[rfm_seg_cols].max() - rfm_profile[rfm_seg_cols].min())
    im = ax.imshow(norm_profile.values, cmap="RdPu", aspect="auto")
    ax.set_xticks(range(len(rfm_seg_cols)))
    ax.set_yticks(range(len(rfm_profile)))
    ax.set_xticklabels(rfm_seg_cols, rotation=45, ha="right")
    ax.set_yticklabels([f"RFM_{idx}" for idx in rfm_profile.index])
    ax.set_title("Heatmap normalisée — 9 segments RFM Sephora", fontweight="bold")
    plt.colorbar(im, ax=ax, label="Valeur normalisée [0-1]")
    fig.tight_layout()
    plt.savefig("../outputs/figures/1_rfm_segments_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n→ Objectif : nos clusters ML doivent être plus granulaires et actionnables que ces 9 segments")
else:
    print("RFM_Segment_ID non disponible dans les features — vérifier notebook 0")
